# NumCompute quickstart

An end-to-end tour of the **NumCompute** toolkit (plain Python + NumPy only). Every section
runs on the same small synthetic customer dataset, which the notebook writes to disk itself, so
you can run all cells top to bottom with no extra files.

| # | Section | Modules |
|---|---|---|
| 1 | Reading CSV files, missing values, custom delimiters, streaming | `io` |
| 2 | Preprocessing, a full `Pipeline`, and evaluation metrics | `preprocessing`, `pipeline`, `metrics` |
| 3 | Top-k, quickselect, binary search, ranking with ties, percentiles | `sort_search`, `rank` |
| 4 | Descriptive statistics, NaN handling, histograms, streaming statistics | `stats` |
| 5 | Finite-difference gradients, Jacobians, gradient checking | `optim` |
| 6 | Loop vs vectorised benchmarks | `benchmarking` |

Run the notebook from the repository root or from `demo/`.

In [1]:
import sys, pathlib

# make `import numcompute` work whether or not the package is installed
HERE = pathlib.Path.cwd().resolve()
ROOT = HERE.parent if HERE.name == "demo" else HERE
sys.path.insert(0, str(ROOT))
DEMO_DIR = ROOT / "demo"
DEMO_DIR.mkdir(exist_ok=True)

import numpy as np
from IPython.display import Markdown, display

import numcompute as nc
import numcompute.io as nio
from numcompute import metrics, optim, rank, sort_search, stats, utils
from numcompute.benchmarking import compare, environment_info, format_table
from numcompute.pipeline import (ColumnSelector, Compose, FeatureUnion,
                                 NearestCentroidClassifier, Pipeline)
from numcompute.preprocessing import Imputer, MinMaxScaler, OneHotEncoder, StandardScaler

np.set_printoptions(precision=3, suppress=True, linewidth=110)
print("NumCompute version:", nc.__version__)
for key, value in environment_info().items():
    print(f"{key:10s} {value}")

NumCompute version: 0.1.0
python     3.15.0
numpy      2.5.4
platform   macOS-26.6.2-arm64-arm-64bit-Mach-O
machine    arm64
processor  arm
cpu_count  8
date       2026-10-23 00:10 UTC


## 1. Reading data with `io`

First we create a synthetic customer dataset (200 rows) and write it to `demo/sample_data.csv`.
About 5-8 % of the entries are deliberately left **missing** (empty fields or `NA`) so we can show
the three missing-value policies.

In [2]:
rng = np.random.default_rng(42)          # fixed seed -> reproducible demo
n = 200
age = rng.integers(18, 70, n).astype(float)
income = np.round(rng.normal(55_000, 15_000, n) + (age - 40) * 400)
city = rng.choice(["Paris", "London", "Rome", "Madrid"], n)
plan = rng.choice(["basic", "plus", "pro"], n, p=[0.5, 0.3, 0.2])
signal = -0.06 * (age - 40) + 0.00005 * (income - 55_000) + 1.5 * (plan == "basic") + rng.normal(0, 0.4, n)
churn = (signal > 0.6).astype(int)      # younger, lower-income "basic"-plan customers churn more

def maybe(value, p, missing=""):
    """Replace a value by a missing marker with probability p."""
    return missing if rng.random() < p else value

lines = ["age,income,city,plan,churn"]
for i in range(n):
    lines.append(",".join([
        maybe(f"{age[i]:.0f}", 0.08),
        maybe(f"{income[i]:.0f}", 0.08),
        maybe(city[i], 0.05),
        maybe(plan[i], 0.05, missing="NA"),
        str(churn[i]),
    ]))

CSV = DEMO_DIR / "sample_data.csv"
TSV = DEMO_DIR / "sample_data.tsv"
CSV.write_text("\n".join(lines) + "\n")
TSV.write_text("\n".join(l.replace(",", "\t") for l in lines) + "\n")   # same data, tab-separated

print("First rows of sample_data.csv:")
print("\n".join(lines[:7]))

First rows of sample_data.csv:
age,income,city,plan,churn
22,53797,London,plus,1
58,48618,London,plus,0
,54128,Paris,basic,1
40,74488,London,pro,0
40,49656,Rome,basic,1
62,74863,Paris,NA,1


In [3]:
# 1a. numeric columns only (age, income); missing entries become NaN by default
X = nio.load_csv(CSV, skip_header=1, usecols=[0, 1])
print("shape:", X.shape, "| dtype:", X.dtype)
print("missing per column [age, income]:", np.isnan(X).sum(axis=0))
print(X[:5])

shape: (200, 2) | dtype: float64
missing per column [age, income]: [15  7]
[[   22. 53797.]
 [   58. 48618.]
 [   nan 54128.]
 [   40. 74488.]
 [   40. 49656.]]


In [4]:
# 1b. the three missing-value policies
X_nan  = nio.load_csv(CSV, skip_header=1, usecols=[0, 1], missing="nan")
X_fill = nio.load_csv(CSV, skip_header=1, usecols=[0, 1], missing="fill", fill_value=-1)
X_skip = nio.load_csv(CSV, skip_header=1, usecols=[0, 1], missing="skip")

print(f"nan  : shape {X_nan.shape},  NaNs left = {np.isnan(X_nan).sum()}")
print(f"fill : shape {X_fill.shape},  NaNs left = {np.isnan(X_fill).sum()},  fill value = -1")
print(f"skip : shape {X_skip.shape}  (rows with a missing value are dropped)")

nan  : shape (200, 2),  NaNs left = 22
fill : shape (200, 2),  NaNs left = 0,  fill value = -1
skip : shape (179, 2)  (rows with a missing value are dropped)


In [5]:
# 1c. custom delimiter: the TSV holds the same data separated by tabs
X_tab = nio.load_csv(TSV, delimiter="\t", skip_header=1, usecols=[0, 1])
print("TSV and CSV give identical arrays:", np.array_equal(X_tab, X_nan, equal_nan=True))

# 1d. dtype handling: integer dtypes refuse to hold missing values unless they are filled
try:
    nio.load_csv(CSV, skip_header=1, usecols=[0, 1], dtype=int)
except ValueError as err:
    print("dtype=int with missing values ->", err)
ints = nio.load_csv(CSV, skip_header=1, usecols=[0, 1], dtype=int, missing="fill", fill_value=0)
print("dtype=int with fill:", ints.dtype, ints[:2].tolist())

# a text column in a numeric load gives a clear error instead of silent garbage
try:
    nio.load_csv(CSV, skip_header=1)
except ValueError as err:
    print("text column in numeric load ->", str(err)[:90], "...")

TSV and CSV give identical arrays: True
dtype=int with missing values -> integer dtype cannot hold missing values; use missing='fill' with an integer fill_value or missing='skip'
dtype=int with fill: int64 [[22, 53797], [58, 48618]]
text column in numeric load -> non-numeric value found in a numeric column: could not convert string to float: np.str_('L ...


In [6]:
# 1e. mixed numeric + categorical columns in one call
X_num, X_cat, num_names, cat_names = nio.load_mixed_csv(CSV, categorical_cols=["city", "plan"])
print("numeric    :", num_names, X_num.shape)
print("categorical:", cat_names, X_cat.shape)
print("first categorical rows:", X_cat[:3].tolist())
print("plan categories incl. the label used for missing entries:", sorted(set(X_cat[:, 1].tolist())))

numeric    : ['age', 'income', 'churn'] (200, 3)
categorical: ['city', 'plan'] (200, 2)
first categorical rows: [['London', 'plus'], ['London', 'plus'], ['Paris', 'basic']]
plan categories incl. the label used for missing entries: ['basic', 'missing', 'plus', 'pro']


In [7]:
# 1f. streaming: read the file in chunks so it never has to fit in memory at once
sizes = [chunk.shape[0] for chunk in nio.iter_csv_chunks(CSV, chunksize=60, skip_header=1, usecols=[0, 1])]
print("chunk sizes:", sizes, "-> total rows:", sum(sizes))
whole = np.vstack(list(nio.iter_csv_chunks(CSV, chunksize=60, skip_header=1, usecols=[0, 1])))
print("chunks reassemble to the full table:", np.array_equal(whole, X_nan, equal_nan=True))

chunk sizes: [60, 60, 60, 20] -> total rows: 200
chunks reassemble to the full table: True


## 2. Preprocessing, `Pipeline` and metrics

We load the file as strings (`dtype=str`; missing entries become the text `"nan"`), then build one
pipeline that sends the **numeric** columns through *Imputer → StandardScaler* and the
**categorical** columns through *OneHotEncoder*, joins them with a `FeatureUnion`, and finishes with
a small classifier that predicts `churn`.

In [8]:
# 2a. the building blocks on their own
age_income = nio.load_csv(CSV, skip_header=1, usecols=[0, 1])           # contains NaN

imputed = Imputer("median").fit_transform(age_income)
print("Imputer(median): NaN before =", np.isnan(age_income).sum(), "| after =", np.isnan(imputed).sum())

scaled = StandardScaler().fit_transform(imputed)
print("StandardScaler : mean =", scaled.mean(axis=0).round(6), "| std =", scaled.std(axis=0).round(6))

minmax = MinMaxScaler(feature_range=(-1, 1)).fit_transform(imputed)
print("MinMaxScaler   : min =", minmax.min(axis=0), "| max =", minmax.max(axis=0))

enc = OneHotEncoder().fit(X_cat)
print("OneHotEncoder  : categories per column =", [c.tolist() for c in enc.categories_])
print("output columns :", enc.get_feature_names_out(["city", "plan"]))

Imputer(median): NaN before = 22 | after = 0
StandardScaler : mean = [-0.  0.] | std = [1. 1.]
MinMaxScaler   : min = [-1. -1.] | max = [1. 1.]
OneHotEncoder  : categories per column = [['London', 'Madrid', 'Paris', 'Rome', 'missing'], ['basic', 'missing', 'plus', 'pro']]
output columns : ['city_London', 'city_Madrid', 'city_Paris', 'city_Rome', 'city_missing', 'plan_basic', 'plan_missing', 'plan_plus', 'plan_pro']


In [9]:
# 2b. special cases handled safely
print("constant column -> zeros, not NaN :", StandardScaler().fit_transform([[5.0], [5.0], [5.0]]).ravel())
print("NaN is ignored when fitting       :", StandardScaler().fit_transform([[1.0], [np.nan], [3.0]]).ravel())

encoder = OneHotEncoder(handle_unknown="ignore").fit([["a"], ["b"]])
print("unknown category -> all zeros     :", encoder.transform([["zzz"]]).ravel())
try:
    OneHotEncoder().fit([["a"], ["b"]]).transform([["zzz"]])
except ValueError as err:
    print("handle_unknown='error' raises     :", err)

try:
    StandardScaler().transform([[1.0]])
except ValueError as err:                       # NotFittedError is a ValueError
    print("using a scaler before fit raises  :", type(err).__name__)

constant column -> zeros, not NaN : [0. 0. 0.]
NaN is ignored when fitting       : [-1. nan  1.]
unknown category -> all zeros     : [0. 0.]
handle_unknown='error' raises     : unknown categories ['zzz'] in column 0
using a scaler before fit raises  : NotFittedError


In [10]:
# 2c. the full pipeline
raw = nio.load_csv(CSV, skip_header=1, dtype=str)       # columns: age, income, city, plan, churn
y = raw[:, 4].astype(int)

perm = np.random.default_rng(0).permutation(len(raw))
train, test = perm[:150], perm[150:]

numeric = Pipeline([
    ("select", ColumnSelector([0, 1], dtype=float)),     # "nan" strings become NaN floats
    ("impute", Imputer("median")),
    ("scale", StandardScaler()),
])
categorical = Pipeline([
    ("select", ColumnSelector([2, 3])),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
model = Pipeline([
    ("features", FeatureUnion([("num", numeric), ("cat", categorical)])),
    ("clf", NearestCentroidClassifier()),
])

model.fit(raw[train], y[train])
predictions = model.predict(raw[test])

Z_train = model["features"].transform(raw[train])
names = ["age", "income"] + list(
    model["features"].transformer_list[1][1]["onehot"].get_feature_names_out(["city", "plan"]))
print("feature matrix:", Z_train.shape, "->", names)
print("any NaN left after the pipeline?", bool(np.isnan(Z_train).any()))
print("first predictions vs truth:", predictions[:10], y[test][:10])

feature matrix: (150, 11) -> ['age', 'income', 'city_London', 'city_Madrid', 'city_Paris', 'city_Rome', 'city_nan', 'plan_basic', 'plan_nan', 'plan_plus', 'plan_pro']
any NaN left after the pipeline? False
first predictions vs truth: [0 1 0 0 0 1 0 1 1 0] [0 1 0 0 0 1 1 0 1 0]


In [11]:
# 2d. evaluation metrics on the held-out 50 rows
y_true = y[test]
print(f"accuracy : {metrics.accuracy(y_true, predictions):.3f}")
print(f"precision: {metrics.precision(y_true, predictions):.3f}   (class 1 = churn)")
print(f"recall   : {metrics.recall(y_true, predictions):.3f}")
print(f"f1       : {metrics.f1(y_true, predictions):.3f}")
print("per-class f1 :", metrics.f1(y_true, predictions, average=None))
print("macro f1     :", round(metrics.f1(y_true, predictions, average="macro"), 3))

cm = metrics.confusion_matrix(y_true, predictions)
print("\nconfusion matrix (rows = true class, columns = predicted class):")
print(cm)

# ROC curve from a score: how much closer a row is to the churn centroid than to the other one
Z_test = model["features"].transform(raw[test])
D = utils.pairwise_distances(Z_test, model["clf"].centroids_)
score = D[:, 0] - D[:, 1]
fpr, tpr, thr = metrics.roc_curve(y_true, score)
print(f"\nROC AUC = {metrics.auc(fpr, tpr):.3f}  (0.5 = chance, 1.0 = perfect), {len(fpr)} curve points")

accuracy : 0.820
precision: 0.840   (class 1 = churn)
recall   : 0.808
f1       : 0.824
per-class f1 : [0.816 0.824]
macro f1     : 0.82

confusion matrix (rows = true class, columns = predicted class):
[[20  4]
 [ 5 21]]

ROC AUC = 0.881  (0.5 = chance, 1.0 = perfect), 51 curve points


In [12]:
# 2e. Compose chains plain functions and transformers, too
log_then_scale = Compose([np.log1p, StandardScaler()])
print(log_then_scale.fit_transform(np.abs(imputed))[:3])
print("steps:", [name for name, _ in log_then_scale.steps])

[[-1.889  0.008]
 [ 0.966 -0.318]
 [ 0.145  0.028]]
steps: ['step0', 'step1']


## 3. `sort_search` and `rank`

### Top-k, quickselect, binary search

In [13]:
# 3a. topk: ties, NaN and extreme k
scores = np.array([7.0, 3.0, 9.0, 3.0, 9.0, np.nan, 5.0])
values, idx = sort_search.topk(scores, 3)
print("top 3 largest :", values, "at indices", idx, " (the tie 9,9 keeps the lower index first)")
values, idx = sort_search.topk(scores, 2, largest=False)
print("2 smallest    :", values, "at indices", idx, " (the tie 3,3 -> lowest index first)")
print("k > n gives everything:", sort_search.topk(scores, 99)[0], "<- NaN is ranked last, never preferred")
print("k = 0:", sort_search.topk(scores, 0)[0], "| values only:", sort_search.topk(scores, 2, return_indices=False))

M = np.random.default_rng(1).integers(0, 100, size=(3, 6))
print("\nrow-wise top 2 of a matrix:\n", M, "\n->", sort_search.topk(M, 2, axis=1)[0].tolist())

top 3 largest : [9. 9. 7.] at indices [2 4 0]  (the tie 9,9 keeps the lower index first)
2 smallest    : [3. 3.] at indices [1 3]  (the tie 3,3 -> lowest index first)
k > n gives everything: [ 9.  9.  7.  5.  3.  3. nan] <- NaN is ranked last, never preferred
k = 0: [] | values only: [9. 9.]

row-wise top 2 of a matrix:
 [[47 51 75 95  3 14]
 [82 94 24 31 86 42]
 [27 82 25 40 64 54]] 
-> [[95, 75], [94, 86], [82, 64]]


In [14]:
# 3b. quickselect: k-th smallest in expected O(n), no full sort
data = np.random.default_rng(2).integers(0, 1000, size=1001)
median = sort_search.quickselect(data, 500)
print("quickselect median:", median, "| np.sort(data)[500]:", np.sort(data)[500])
print("minimum:", sort_search.quickselect(data, 0), "| maximum:", sort_search.quickselect(data, 1000))
print("all-equal input works:", sort_search.quickselect(np.full(10_000, 7), 5000))
try:
    sort_search.quickselect([1.0, np.nan, 3.0], 1)
except ValueError as err:
    print("NaN input ->", err)

quickselect median: 499 | np.sort(data)[500]: 499
minimum: 1 | maximum: 999
all-equal input works: 7
NaN input -> a contains NaN; remove or impute it first


In [15]:
# 3c. binary_search returns (insertion index, found)
arr = [1, 3, 3, 3, 5, 8]
for x in (3, 4, 0, 9):
    print(f"x = {x}:  left -> {sort_search.binary_search(arr, x)}   right -> {sort_search.binary_search(arr, x, side='right')}")
idx, found = sort_search.binary_search_many(arr, [0, 3, 4, 9])
print("many at once:", idx.tolist(), found.tolist())

# 3d. stable multi-key sort: department ascending, salary descending
table = np.array([[2, 50], [1, 70], [2, 90], [1, 70], [1, 40]])
print("\nmulti_key_sort:\n", sort_search.multi_key_sort(table, keys=[0, 1], ascending=[True, False]))
print("stable_argsort keeps tie order:", sort_search.stable_argsort([3, 1, 2, 1, 3]), "| descending:",
      sort_search.stable_argsort([3, 1, 2, 1, 3], descending=True))

x = 3:  left -> (1, True)   right -> (4, True)
x = 4:  left -> (4, False)   right -> (4, False)
x = 0:  left -> (0, False)   right -> (0, False)
x = 9:  left -> (6, False)   right -> (6, False)
many at once: [0, 1, 4, 6] [False, True, False, False]

multi_key_sort:
 [[ 1 70]
 [ 1 70]
 [ 1 40]
 [ 2 90]
 [ 2 50]]
stable_argsort keeps tie order: [1 3 2 0 4] | descending: [0 4 2 1 3]


### Ranking with ties and percentiles

In [16]:
# 3e. the five tie-handling methods on the same data
x = np.array([10, 20, 20, 30, 20, 5])
print("values :", x)
for method in ("average", "min", "max", "dense", "ordinal"):
    print(f"{method:8s}:", rank.rank(x, method=method))

print("\nNaN policies on [3, nan, 1, 3]:")
print("  omit      :", rank.rank([3, np.nan, 1, 3]))
print("  propagate :", rank.rank([3, np.nan, 1, 3], nan_policy="propagate"))
print("column-wise ranks of a matrix (axis=0):\n", rank.rank([[3, 1], [1, 1], [2, 5]], axis=0))

values : [10 20 20 30 20  5]
average : [2. 4. 4. 6. 4. 1.]
min     : [2. 3. 3. 6. 3. 1.]
max     : [2. 5. 5. 6. 5. 1.]
dense   : [2. 3. 3. 4. 3. 1.]
ordinal : [2. 3. 4. 6. 5. 1.]

NaN policies on [3, nan, 1, 3]:
  omit      : [2.5 nan 1.  2.5]
  propagate : [nan nan nan nan]
column-wise ranks of a matrix (axis=0):
 [[3.  1.5]
 [1.  1.5]
 [2.  3. ]]


In [17]:
# 3f. percentiles with the four interpolation rules (+ nearest), checked against NumPy
sample = np.random.default_rng(3).normal(50, 10, size=25).round(1)
q = [5, 25, 50, 75, 95]
print("q                :", q)
for how in ("linear", "lower", "higher", "midpoint", "nearest"):
    mine = rank.percentile(sample, q, interpolation=how)
    ref = np.percentile(sample, q, method=how)
    print(f"{how:9s}", mine, "| max |diff| vs NumPy =", float(np.abs(mine - ref).max()))

with_nan = np.array([1.0, np.nan, 3.0, 5.0])
print("\nmedian with NaN  propagate:", rank.percentile(with_nan, 50),
      "| omit:", rank.percentile(with_nan, 50, nan_policy="omit"))

q                : [5, 25, 50, 75, 95]
linear    [31.72 44.9  47.7  54.2  69.42] | max |diff| vs NumPy = 0.0
lower     [29.8 44.9 47.7 54.2 65.5] | max |diff| vs NumPy = 0.0
higher    [39.4 44.9 47.7 54.2 70.4] | max |diff| vs NumPy = 0.0
midpoint  [34.6  44.9  47.7  54.2  67.95] | max |diff| vs NumPy = 0.0
nearest   [29.8 44.9 47.7 54.2 70.4] | max |diff| vs NumPy = 0.0

median with NaN  propagate: nan | omit: 3.0


## 4. `stats`

Statistics use the age and income columns, which still contain missing values.

In [18]:
# 4a. NaN handling: propagate (default) vs omit
print("mean with NaN, propagate:", stats.mean(age_income, axis=0))
print("mean with NaN, omit     :", stats.mean(age_income, axis=0, nan_policy="omit"))
print("std  (omit)             :", stats.std(age_income, axis=0, nan_policy="omit"))
try:
    stats.mean(age_income, nan_policy="raise")
except ValueError as err:
    print("nan_policy='raise'      ->", err)

summary = stats.describe(age_income, axis=0, nan_policy="omit")
print("\nsummary of [age, income]:")
for key, value in summary.items():
    print(f"  {key:7s}", value)

mean with NaN, propagate: [nan nan]
mean with NaN, omit     : [   44.135 56053.487]
std  (omit)             : [   14.011 15880.955]
nan_policy='raise'      -> a contains NaN

summary of [age, income]:
  count   [185 193]
  mean    [   44.135 56053.487]
  std     [   14.011 15880.955]
  min     [  19. 9300.]
  q25     [   33. 44921.]
  median  [   44. 55188.]
  q75     [   57. 66691.]
  max     [    69. 107376.]


In [19]:
# 4b. quantiles with NaN
ages = age_income[:, 0]
print("age quantiles  q=[0.1, 0.5, 0.9]  omit     :", stats.quantile(ages, [0.1, 0.5, 0.9], nan_policy="omit"))
print("                                   propagate:", stats.quantile(ages, [0.1, 0.5, 0.9]))
print("column-wise income quartiles:", stats.quantile(age_income, [0.25, 0.75], axis=0, nan_policy="omit").T.tolist())

age quantiles  q=[0.1, 0.5, 0.9]  omit     : [23.4 44.  62. ]
                                   propagate: [nan nan nan]
column-wise income quartiles: [[33.0, 57.0], [44921.0, 66691.0]]


In [20]:
# 4c. histogram (NaNs are ignored) with a text bar chart
counts, edges = stats.histogram(ages, bins=8)
for c, lo, hi in zip(counts, edges[:-1], edges[1:]):
    print(f"[{lo:5.1f}, {hi:5.1f})  {c:3d}  {'#' * int(c)}")
print("total counted:", counts.sum(), "of", int((~np.isnan(ages)).sum()), "non-missing ages")

density, e = stats.histogram(ages, bins=8, density=True)
print("density integrates to", round(float((density * np.diff(e)).sum()), 6))

[ 19.0,  25.2)   25  #########################
[ 25.2,  31.5)   16  ################
[ 31.5,  37.8)   21  #####################
[ 37.8,  44.0)   29  #############################
[ 44.0,  50.2)   22  ######################
[ 50.2,  56.5)   24  ########################
[ 56.5,  62.8)   32  ################################
[ 62.8,  69.0)   16  ################
total counted: 185 of 185 non-missing ages
density integrates to 1.0


In [21]:
# 4d. streaming mean/variance (Welford/Chan): process the CSV chunk by chunk
running = stats.RunningStats()
for chunk in nio.iter_csv_chunks(CSV, chunksize=40, skip_header=1, usecols=[0, 1]):
    running.update(chunk)                      # state stays O(n_features), NaNs are skipped

print("streamed count :", running.count)
print("streamed mean  :", running.mean, "| in-memory :", stats.mean(age_income, axis=0, nan_policy="omit"))
print("streamed var   :", running.var(), "| in-memory :", stats.var(age_income, axis=0, nan_policy="omit"))

# why it matters: the textbook formula E[x^2] - E[x]^2 loses precision when values are large
big = 1e9 + np.random.default_rng(4).normal(size=100_000)
naive = np.mean(big ** 2) - np.mean(big) ** 2
print("\ntrue variance is about 1.0")
print("  naive formula    :", naive)
print("  np.var           :", np.var(big))
print("  RunningStats     :", float(stats.RunningStats().update(big).var()[0]))

streamed count : [185. 193.]
streamed mean  : [   44.135 56053.487] | in-memory : [   44.135 56053.487]
streamed var   : [1.963e+02 2.522e+08] | in-memory : [1.963e+02 2.522e+08]

true variance is about 1.0
  naive formula    : -128.0
  np.var           : 0.9981538437667178
  RunningStats     : 0.9981538437667178


## 5. `optim`: finite-difference gradients and Jacobians

For a scalar function $f(x_0,x_1)=x_0^2+3x_1+\sin(x_0x_1)$ the exact gradient is
$\left(2x_0+x_1\cos(x_0x_1),\; 3+x_0\cos(x_0x_1)\right)$.

In [22]:
f = lambda v: v[0] ** 2 + 3 * v[1] + np.sin(v[0] * v[1])
exact_grad = lambda v: np.array([2 * v[0] + v[1] * np.cos(v[0] * v[1]), 3 + v[0] * np.cos(v[0] * v[1])])
x0 = np.array([1.0, 2.0])

print("exact   :", exact_grad(x0))
for method in ("central", "forward", "backward"):
    g = optim.grad(f, x0, method=method)
    print(f"{method:8s}:", g, "| max error", f"{np.abs(g - exact_grad(x0)).max():.2e}")

exact   : [1.168 2.584]
central : [1.168 2.584] | max error 6.52e-11
forward : [1.168 2.584] | max error 8.19e-06
backward: [1.168 2.584] | max error 8.19e-06


In [23]:
# 5b. accuracy depends on the step size h: truncation error falls, then rounding error takes over
print(f"{'h':>8} | {'central error':>14} | {'forward error':>14}")
for h in (1e-1, 1e-2, 1e-3, 1e-5, 1e-8, 1e-11):
    ec = np.abs(optim.grad(f, x0, h=h) - exact_grad(x0)).max()
    ef = np.abs(optim.grad(f, x0, h=h, method="forward") - exact_grad(x0)).max()
    print(f"{h:8.0e} | {ec:14.2e} | {ef:14.2e}")
print("\ncentral differences are O(h^2) accurate, forward are O(h); the default h=1e-5 is a good balance.")

       h |  central error |  forward error
   1e-01 |       5.54e-03 |       7.57e-02
   1e-02 |       5.55e-05 |       8.13e-03
   1e-03 |       5.55e-07 |       8.18e-04
   1e-05 |       6.52e-11 |       8.19e-06
   1e-08 |       1.24e-08 |       3.97e-08
   1e-11 |       1.82e-05 |       5.33e-05

central differences are O(h^2) accurate, forward are O(h); the default h=1e-5 is a good balance.


In [24]:
# 5c. Jacobian of a vector function F: R^2 -> R^3
F = lambda v: np.array([v[0] * v[1], v[0] + v[1] ** 2, np.sin(v[0])])
exact_J = lambda v: np.array([[v[1], v[0]], [1.0, 2 * v[1]], [np.cos(v[0]), 0.0]])

J = optim.jacobian(F, x0)
print("Jacobian shape:", J.shape, "(rows = outputs, columns = inputs)")
print(J)
print("max error central:", f"{np.abs(J - exact_J(x0)).max():.2e}",
      "| forward:", f"{np.abs(optim.jacobian(F, x0, method='forward') - exact_J(x0)).max():.2e}")

# a linear map is recovered exactly
A = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print("\nJacobian of x -> A @ x equals A:", np.allclose(optim.jacobian(lambda v: A @ v, np.ones(3)), A, atol=1e-7))

Jacobian shape: (3, 2) (rows = outputs, columns = inputs)
[[2.   1.  ]
 [1.   4.  ]
 [0.54 0.  ]]
max error central: 3.79e-11 | forward: 1.00e-05

Jacobian of x -> A @ x equals A: True


In [25]:
# 5d. gradient checking: validate a hand-written gradient against finite differences
print("correct gradient  ->", f"{optim.gradient_check(f, exact_grad, x0):.2e}   (tiny = good)")
wrong = lambda v: exact_grad(v) * np.array([1.0, 1.1])        # deliberate 10 % mistake in one component
print("buggy gradient    ->", f"{optim.gradient_check(f, wrong, x0):.2e}   (large = bug detected)")

# 5e. batched evaluation: one call to f instead of 2n calls
def f_batch(V):
    V = np.atleast_2d(V)
    return V[:, 0] ** 2 + 3 * V[:, 1] + np.sin(V[:, 0] * V[:, 1])
print("\nvectorized=True gives the same gradient:", np.allclose(optim.grad(f_batch, x0, vectorized=True), optim.grad(f, x0)))

# 5f. a few steps of gradient descent with Armijo backtracking line search
bowl = lambda v: float((v[0] - 3) ** 2 + 2 * (v[1] + 1) ** 2)
point = np.array([0.0, 0.0])
for step in range(20):
    g = optim.grad(bowl, point)
    if np.linalg.norm(g) < 1e-6:                      # stop once the gradient has vanished
        print(f"converged after {step} steps: x = {point}  (true minimum is [3, -1])")
        break
    alpha = optim.backtracking_line_search(bowl, point, -g, grad_x=g)
    point = point - alpha * g
    print(f"step {step}: x = {point}, f = {bowl(point):.6f}, step length = {alpha}")

correct gradient  -> 1.56e-11   (tiny = good)
buggy gradient    -> 4.37e-02   (large = bug detected)

vectorized=True gives the same gradient: True
step 0: x = [ 3. -2.], f = 2.000000, step length = 0.5
step 1: x = [ 3. -1.], f = 0.000000, step length = 0.25
converged after 2 steps: x = [ 3. -1.]  (true minimum is [3, -1])


## 6. Benchmarks: Python loops vs vectorised NumCompute

`compare` runs both implementations on identical, seeded data, verifies that they return the same
result (`Match` column) and reports the best time of several repeats. The full suite (eight
comparisons, larger sizes) is in `benchmark/run_benchmarks.py`.

In [26]:
import math

def standardize_loop(X):
    n, d = X.shape
    out = np.empty((n, d))
    for j in range(d):
        col = X[:, j].tolist()
        mu = sum(col) / n
        sd = math.sqrt(sum((v - mu) ** 2 for v in col) / n) or 1.0
        for i in range(n):
            out[i, j] = (col[i] - mu) / sd
    return out

def softmax_loop(X):
    out = []
    for row in X.tolist():
        m = max(row)
        e = [math.exp(v - m) for v in row]
        t = sum(e)
        out.append([v / t for v in e])
    return np.array(out)

def pairwise_loop(A, B):
    return np.array([[math.sqrt(sum((p - q) ** 2 for p, q in zip(a, b))) for b in B.tolist()] for a in A.tolist()])

def rank_loop(x):
    xs = x.tolist()
    return np.array([sum(w < v for w in xs) + (sum(w == v for w in xs) + 1) / 2 for v in xs])

def topk_loop(x, k):
    xs = x.tolist()
    order = sorted(range(len(xs)), key=lambda i: (-xs[i], i))[:k]
    return np.array([xs[i] for i in order])

def histogram_loop(x, bins):
    lo, hi = min(x.tolist()), max(x.tolist())
    width = (hi - lo) / bins
    counts = [0] * bins
    for v in x.tolist():
        counts[min(int((v - lo) / width), bins - 1)] += 1
    return np.array(counts)

cases = [
    ("Standardise 10 columns", standardize_loop, lambda X: StandardScaler().fit_transform(X),
     lambda n, r: (r.normal(size=(n, 10)),), [500, 2000, 8000]),
    ("Row-wise softmax (20 classes)", softmax_loop, utils.softmax,
     lambda n, r: (r.normal(size=(n, 20)),), [500, 2000, 8000]),
    ("Pairwise Euclidean distances", pairwise_loop, utils.pairwise_distances,
     lambda n, r: (r.normal(size=(n, 8)), r.normal(size=(n, 8))), [50, 100, 200]),
    ("Average ranks with ties", rank_loop, rank.rank,
     lambda n, r: (r.integers(0, n // 2, size=n).astype(float),), [300, 600, 1200]),
    ("Top-10 of n values", topk_loop, lambda x, k: sort_search.topk(x, k, return_indices=False),
     lambda n, r: (r.normal(size=n), 10), [10_000, 50_000, 200_000]),
    ("Histogram (20 bins)", histogram_loop, lambda x, b: stats.histogram(x, bins=b)[0],
     lambda n, r: (r.normal(size=n), 20), [10_000, 50_000, 200_000]),
]

results = {}
for title, loop_fn, vec_fn, make_args, sizes in cases:
    results[title] = compare(loop_fn, vec_fn, make_args, sizes, repeats=3, seed=0)
    display(Markdown(format_table(results[title], title=title)))

**Standardise 10 columns**

| Size | Loop | Vectorised | Speed-up | Match |
|---:|---:|---:|---:|:---:|
| 500 | 740.5 us | 80.3 us | 9.2x | yes |
| 2,000 | 3.03 ms | 189.1 us | 16.0x | yes |
| 8,000 | 12.16 ms | 590.8 us | 20.6x | yes |

**Row-wise softmax (20 classes)**

| Size | Loop | Vectorised | Speed-up | Match |
|---:|---:|---:|---:|:---:|
| 500 | 1.13 ms | 82.3 us | 13.7x | yes |
| 2,000 | 4.74 ms | 329.0 us | 14.4x | yes |
| 8,000 | 20.11 ms | 1.09 ms | 18.4x | yes |

**Pairwise Euclidean distances**

| Size | Loop | Vectorised | Speed-up | Match |
|---:|---:|---:|---:|:---:|
| 50 | 2.36 ms | 17.7 us | 133.5x | yes |
| 100 | 9.68 ms | 33.3 us | 291.1x | yes |
| 200 | 39.20 ms | 83.3 us | 470.7x | yes |

**Average ranks with ties**

| Size | Loop | Vectorised | Speed-up | Match |
|---:|---:|---:|---:|:---:|
| 300 | 3.19 ms | 51.0 us | 62.5x | yes |
| 600 | 12.76 ms | 46.8 us | 272.5x | yes |
| 1,200 | 50.38 ms | 71.1 us | 708.7x | yes |

**Top-10 of n values**

| Size | Loop | Vectorised | Speed-up | Match |
|---:|---:|---:|---:|:---:|
| 10,000 | 2.55 ms | 150.1 us | 17.0x | yes |
| 50,000 | 15.43 ms | 638.5 us | 24.2x | yes |
| 200,000 | 96.75 ms | 2.53 ms | 38.2x | yes |

**Histogram (20 bins)**

| Size | Loop | Vectorised | Speed-up | Match |
|---:|---:|---:|---:|:---:|
| 10,000 | 1.10 ms | 251.1 us | 4.4x | yes |
| 50,000 | 5.81 ms | 1.31 ms | 4.4x | yes |
| 200,000 | 23.36 ms | 4.93 ms | 4.7x | yes |

In [27]:
# performance summary: largest size of every comparison
rows = ["| Operation | Largest size | Python loop | Vectorised | Speed-up | Same result |",
        "|---|---:|---:|---:|---:|:---:|"]
for title, table in results.items():
    last = table[-1]
    fmt = lambda s: f"{s * 1e3:.2f} ms" if s >= 1e-3 else f"{s * 1e6:.1f} us"
    rows.append(f"| {title} | {last['size']:,} | {fmt(last['loop_s'])} | {fmt(last['vec_s'])} "
                f"| **{last['speedup']:.0f}x** | {'yes' if last['match'] else 'NO'} |")
display(Markdown("\n".join(rows)))
print("every vectorised result matches its loop version:", all(r["match"] for t in results.values() for r in t))

| Operation | Largest size | Python loop | Vectorised | Speed-up | Same result |
|---|---:|---:|---:|---:|:---:|
| Standardise 10 columns | 8,000 | 12.16 ms | 590.8 us | **21x** | yes |
| Row-wise softmax (20 classes) | 8,000 | 20.11 ms | 1.09 ms | **18x** | yes |
| Pairwise Euclidean distances | 200 | 39.20 ms | 83.3 us | **471x** | yes |
| Average ranks with ties | 1,200 | 50.38 ms | 71.1 us | **709x** | yes |
| Top-10 of n values | 200,000 | 96.75 ms | 2.53 ms | **38x** | yes |
| Histogram (20 bins) | 200,000 | 23.36 ms | 4.93 ms | **5x** | yes |

every vectorised result matches its loop version: True


## Summary

* **`io`**: CSV/TSV loading with fill/skip/NaN policies, typed columns, mixed data and streaming.
* **`preprocessing` + `pipeline`**: imputing, scaling and one-hot encoding chained in one `Pipeline`,
  evaluated with `metrics` (accuracy, precision, recall, F1, confusion matrix, ROC AUC).
* **`sort_search` + `rank`**: deterministic tie and NaN handling in top-k, quickselect, binary search,
  ranking and percentiles.
* **`stats`**: NaN-aware descriptive statistics, quantiles, histograms, numerically stable streaming statistics.
* **`optim`**: finite-difference gradients/Jacobians, gradient checking and line search.
* **`benchmarking`**: every vectorised routine returns the same result as its Python-loop counterpart,
  and runs many times faster on non-trivial sizes.

Run the test-suite with `python -m pytest tests -q` and the full benchmark with
`python benchmark/run_benchmarks.py`.